In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
import joblib

df = pd.read_csv("cleaned_data.csv")

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(max_depth=5)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(classification_report(y_test, pred))



              precision    recall  f1-score   support

         0.0       0.94      0.74      0.82       102
         1.0       0.78      0.95      0.86       103

    accuracy                           0.84       205
   macro avg       0.86      0.84      0.84       205
weighted avg       0.86      0.84      0.84       205



In [ ]:
joblib.dump(model, "heart_model.pkl")

['heart_model.pkl']

In [ ]:
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

model = joblib.load("heart_model.pkl")

# Original raw sample data provided by the user. It has 12 features.
# The model expects 13 features, meaning one feature is missing from the sample.
# Based on the original DataFrame columns, 'restecg' is likely the missing feature.
# We assume a default value of 0 for 'restecg' for this sample.
# Corrected raw sample with 13 features:
# ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
raw_sample_values = [52, 1, 0, 125, 212, 0, 0, 168, 0, 1.0, 2, 2, 3] # Added 0 for 'restecg'

# To properly normalize the new sample, we need to use the scaler that was fitted
# on the original training data. The scaler in cell 'tv6ApIiJ8cYM' was fitted
# on the entire raw DataFrame (including the 'target' column) after filling NaNs.
# We need to recreate that scaler.

# 1. Load the original raw dataset to fit the scaler
original_df = pd.read_csv("/content/heart.csv")

# 2. Fill missing values in the original_df, replicating the preprocessing step
original_df.fillna(original_df.mean(numeric_only=True), inplace=True)

# 3. Instantiate and fit the scaler on the original (cleaned) entire DataFrame
scaler = MinMaxScaler()
scaler.fit(original_df[original_df.columns])

# Define the feature columns (excluding 'target') in the order the model expects
feature_cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

# Create a temporary DataFrame for the sample values, including a dummy 'target' column.
# This is necessary because the original scaler was fitted on a DataFrame with a 'target' column.
# The order of columns must match the original DataFrame for the scaler to work correctly.
# The dummy 'target' value (e.g., 0) does not affect scaling of feature columns.
original_df_columns_order = original_df.columns.tolist()

sample_df_for_scaling = pd.DataFrame([raw_sample_values + [0]], columns=feature_cols + ['target'])

# Transform the sample using the fitted scaler
scaled_sample_array = scaler.transform(sample_df_for_scaling[original_df_columns_order])

# The model expects 13 features (X, without 'target').
# So, we take the first 13 elements from the scaled array (dropping the scaled 'target' column).
scaled_sample = scaled_sample_array[:, :-1]

prediction = model.predict(scaled_sample)

print("Prediction:", prediction)


Prediction: [0.]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
